# Model Export for Deployment

This notebook demonstrates how to export trained models in various formats for deployment.

In [3]:
import torch
import torch.nn as nn
from torchvision import models
import onnx
import onnxruntime as ort
import numpy as np
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"ONNX version: {onnx.__version__}")
print(f"ONNX Runtime version: {ort.__version__}")

Using device: cpu
PyTorch version: 2.14.0+cpu
ONNX version: 1.22.0
ONNX Runtime version: 1.29.0


In [4]:
# Load trained models
def load_model(model_name, num_classes=102):
    if model_name == 'vgg16':
        model = models.vgg16(weights=None)
        model.classifier[6] = nn.Linear(4096, num_classes)
        model.load_state_dict(torch.load('../models/vgg16_flowers102.pth', map_location=device))
    elif model_name == 'resnet50':
        model = models.resnet50(weights=None)
        model.fc = nn.Linear(2048, num_classes)
        model.load_state_dict(torch.load('../models/resnet50_flowers102.pth', map_location=device))
    
    model = model.to(device)
    model.eval()
    return model

vgg16_model = load_model('vgg16')
resnet50_model = load_model('resnet50')
print("Models loaded successfully!")

Models loaded successfully!


In [5]:
# Export to PyTorch Script format (TorchScript)
def export_torchscript(model, model_name, save_path='../models/'):
    """Export model to TorchScript format"""
    model.eval()
    
    # Create dummy input
    dummy_input = torch.randn(1, 3, 224, 224).to(device)
    
    # Trace the model
    traced_model = torch.jit.trace(model, dummy_input)
    
    # Save
    save_path = Path(save_path) / f'{model_name}_traced.pt'
    traced_model.save(save_path)
    print(f"{model_name} TorchScript saved to: {save_path}")
    
    # Verify
    loaded_model = torch.jit.load(save_path)
    with torch.no_grad():
        original_output = model(dummy_input)
        traced_output = loaded_model(dummy_input)
    
    max_diff = torch.max(torch.abs(original_output - traced_output)).item()
    print(f"  Max difference between original and traced: {max_diff:.6f}")
    
    return traced_model

print("Exporting VGG16 to TorchScript...")
vgg16_traced = export_torchscript(vgg16_model, 'vgg16')

print("\nExporting ResNet50 to TorchScript...")
resnet50_traced = export_torchscript(resnet50_model, 'resnet50')

Exporting VGG16 to TorchScript...
vgg16 TorchScript saved to: ..\models\vgg16_traced.pt
  Max difference between original and traced: 0.000000

Exporting ResNet50 to TorchScript...
resnet50 TorchScript saved to: ..\models\resnet50_traced.pt
  Max difference between original and traced: 0.000000


In [7]:
# Export to ONNX format
def export_onnx(model, model_name, save_path='../models/'):
    """Export model to ONNX format"""
    model.eval()
    
    # Create dummy input
    dummy_input = torch.randn(1, 3, 224, 224).to(device)
    
    # Export
    onnx_path = Path(save_path) / f'{model_name}.onnx'
    torch.onnx.export(
        model,
        dummy_input,
        onnx_path,
        export_params=True,
        opset_version=12,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
    )
    
    print(f"{model_name} ONNX saved to: {onnx_path}")
    
    # Verify ONNX model
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print(f"  ONNX model is valid!")
    
    # Test with ONNX Runtime
    ort_session = ort.InferenceSession(str(onnx_path))
    
    # Prepare input
    ort_inputs = {ort_session.get_inputs()[0].name: dummy_input.cpu().numpy()}
    
    # Run inference
    ort_outputs = ort_session.run(None, ort_inputs)
    
    # Compare with PyTorch output
    with torch.no_grad():
        pytorch_output = model(dummy_input)
    
    max_diff = np.max(np.abs(pytorch_output.cpu().numpy() - ort_outputs[0]))
    print(f"  Max difference between PyTorch and ONNX: {max_diff:.6f}")
    
    return onnx_path

print("Exporting VGG16 to ONNX...")
vgg16_onnx_path = export_onnx(vgg16_model, 'vgg16')

print("\nExporting ResNet50 to ONNX...")
resnet50_onnx_path = export_onnx(resnet50_model, 'resnet50')

Exporting VGG16 to ONNX...


W0907 22:45:50.565000 6292 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 12 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `VGG([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `VGG([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 12).
Failed to convert the model to the target version 12 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "c:\Users\user\Desktop\transfer-learning-images\.venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "c:\Users\user\Desktop\transfer-learning-images\.venv\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "c:\Users\user\Desktop\transfer-learning-images\.venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
vgg16 ONNX saved to: ..\models\vgg16.onnx
  ONNX model is valid!


...

  Max difference between PyTorch and ONNX: 0.000002

Exporting ResNet50 to ONNX...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 12).


[torch.onnx] Translate the graph into ONNX... ✅


Failed to convert the model to the target version 12 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "c:\Users\user\Desktop\transfer-learning-images\.venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "c:\Users\user\Desktop\transfer-learning-images\.venv\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "c:\Users\user\Desktop\transfer-learning-images\.venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "c:\Users\user\Desktop\transfer-learning-images\.venv\Lib\

[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
resnet50 ONNX saved to: ..\models\resnet50.onnx
  ONNX model is valid!
  Max difference between PyTorch and ONNX: 0.000003


In [8]:
# Optimize ONNX models
from onnxruntime.transformers import optimizer

def optimize_onnx_model(onnx_path, optimized_path):
    """Optimize ONNX model for inference"""
    # Load and optimize
    optimized_model = optimizer.optimize_model(
        onnx_path,
        model_type='bert',  # Use appropriate type
        num_heads=0,
        hidden_size=0
    )
    
    # Save optimized model
    optimized_model.save_model_to_file(optimized_path)
    print(f"Optimized model saved to: {optimized_path}")
    
    # Compare file sizes
    original_size = Path(onnx_path).stat().st_size / (1024 * 1024)
    optimized_size = Path(optimized_path).stat().st_size / (1024 * 1024)
    print(f"  Original size: {original_size:.2f} MB")
    print(f"  Optimized size: {optimized_size:.2f} MB")
    print(f"  Size reduction: {(1 - optimized_size/original_size)*100:.1f}%")

# Optimize both models
print("Optimizing VGG16 ONNX model...")
optimize_onnx_model(str(vgg16_onnx_path), '../models/vgg16_optimized.onnx')

print("\nOptimizing ResNet50 ONNX model...")
optimize_onnx_model(str(resnet50_onnx_path), '../models/resnet50_optimized.onnx')

Optimizing VGG16 ONNX model...


shape inference failed which might impact useless cast node detection.
symbolic shape inference disabled or failed.
symbolic shape inference disabled or failed.


Optimized model saved to: ../models/vgg16_optimized.onnx
  Original size: 0.06 MB
  Optimized size: 513.77 MB
  Size reduction: -875644.5%

Optimizing ResNet50 ONNX model...


shape inference failed which might impact useless cast node detection.
symbolic shape inference disabled or failed.
symbolic shape inference disabled or failed.


Optimized model saved to: ../models/resnet50_optimized.onnx
  Original size: 0.23 MB
  Optimized size: 90.40 MB
  Size reduction: -39063.1%


In [9]:
# Export model metadata
def export_model_metadata(model_name, model, save_path='../models/'):
    """Export model metadata as JSON"""
    metadata = {
        'model_name': model_name,
        'architecture': model.__class__.__name__,
        'num_parameters': sum(p.numel() for p in model.parameters()),
        'num_trainable_parameters': sum(p.numel() for p in model.parameters() if p.requires_grad),
        'input_size': [3, 224, 224],
        'num_classes': 102,
        'framework': 'PyTorch',
        'pretrained': 'ImageNet',
        'dataset': 'Oxford 102 Flowers'
    }
    
    json_path = Path(save_path) / f'{model_name}_metadata.json'
    with open(json_path, 'w') as f:
        json.dump(metadata, f, indent=4)
    
    print(f"{model_name} metadata saved to: {json_path}")
    return metadata

print("Exporting VGG16 metadata...")
vgg16_metadata = export_model_metadata('vgg16', vgg16_model)

print("\nExporting ResNet50 metadata...")
resnet50_metadata = export_model_metadata('resnet50', resnet50_model)

print("\nMetadata content:")
print(json.dumps(vgg16_metadata, indent=2))

Exporting VGG16 metadata...
vgg16 metadata saved to: ..\models\vgg16_metadata.json

Exporting ResNet50 metadata...
resnet50 metadata saved to: ..\models\resnet50_metadata.json

Metadata content:
{
  "model_name": "vgg16",
  "architecture": "VGG",
  "num_parameters": 134678438,
  "num_trainable_parameters": 134678438,
  "input_size": [
    3,
    224,
    224
  ],
  "num_classes": 102,
  "framework": "PyTorch",
  "pretrained": "ImageNet",
  "dataset": "Oxford 102 Flowers"
}


In [10]:
# Benchmark different model formats
import time

def benchmark_formats(pytorch_model, traced_model, onnx_path, num_runs=100):
    """Benchmark inference speed across formats"""
    dummy_input = torch.randn(1, 3, 224, 224).to(device)
    
    # PyTorch
    start = time.time()
    with torch.no_grad():
        for _ in range(num_runs):
            pytorch_model(dummy_input)
    pytorch_time = (time.time() - start) / num_runs
    
    # TorchScript
    start = time.time()
    with torch.no_grad():
        for _ in range(num_runs):
            traced_model(dummy_input)
    traced_time = (time.time() - start) / num_runs
    
    # ONNX Runtime
    ort_session = ort.InferenceSession(str(onnx_path))
    ort_input = dummy_input.cpu().numpy()
    
    start = time.time()
    for _ in range(num_runs):
        ort_session.run(None, {ort_session.get_inputs()[0].name: ort_input})
    onnx_time = (time.time() - start) / num_runs
    
    print(f"PyTorch: {pytorch_time*1000:.2f} ms/inference")
    print(f"TorchScript: {traced_time*1000:.2f} ms/inference")
    print(f"ONNX Runtime: {onnx_time*1000:.2f} ms/inference")
    print(f"\nSpeedup (ONNX vs PyTorch): {pytorch_time/onnx_time:.2f}x")
    print(f"Speedup (TorchScript vs PyTorch): {pytorch_time/traced_time:.2f}x")

print("Benchmarking VGG16 formats...")
benchmark_formats(vgg16_model, vgg16_traced, vgg16_onnx_path)

print("\nBenchmarking ResNet50 formats...")
benchmark_formats(resnet50_model, resnet50_traced, resnet50_onnx_path)

Benchmarking VGG16 formats...
PyTorch: 686.63 ms/inference
TorchScript: 791.79 ms/inference
ONNX Runtime: 378.11 ms/inference

Speedup (ONNX vs PyTorch): 1.82x
Speedup (TorchScript vs PyTorch): 0.87x

Benchmarking ResNet50 formats...
PyTorch: 302.26 ms/inference
TorchScript: 224.45 ms/inference
ONNX Runtime: 90.15 ms/inference

Speedup (ONNX vs PyTorch): 3.35x
Speedup (TorchScript vs PyTorch): 1.35x


In [11]:
# Summary
print("=" * 60)
print("MODEL EXPORT SUMMARY")
print("=" * 60)
print("\nExported Models:")
models_dir = Path('../models')
for model_file in sorted(models_dir.glob('*')):
    size_mb = model_file.stat().st_size / (1024 * 1024)
    print(f"  - {model_file.name}: {size_mb:.2f} MB")
print("\n✓ Models exported successfully in multiple formats")
print("✓ Ready for deployment")
print("=" * 60)

MODEL EXPORT SUMMARY

Exported Models:
  - evaluation_results.pth: 0.14 MB
  - resnet50.onnx: 0.23 MB
  - resnet50.onnx.data: 90.38 MB
  - resnet50_flowers102.pth: 90.78 MB
  - resnet50_metadata.json: 0.00 MB
  - resnet50_optimized.onnx: 90.40 MB
  - resnet50_traced.pt: 91.05 MB
  - vgg16.onnx: 0.06 MB
  - vgg16.onnx.data: 513.81 MB
  - vgg16_flowers102.pth: 513.77 MB
  - vgg16_metadata.json: 0.00 MB
  - vgg16_optimized.onnx: 513.77 MB
  - vgg16_traced.pt: 513.84 MB

✓ Models exported successfully in multiple formats
✓ Ready for deployment
